In [1]:
from transformers import AutoModel, AutoTokenizer
import torch

model = AutoModel.from_pretrained("facebook/mms-tts-kan")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-kan")

In [2]:
import os
import pandas as pd
import torchaudio
import librosa
import numpy as np
from jiwer import wer, cer


metadata = [
    ("ID", "Text"),
    ("train_kannadafullmale_00001", "ಸಮಕಾಲೀನ ಐತಿಹಾಸಿಕ ಸಾಹಸ ಮತ್ತು ಪೌರಾಣಿಕ ಸಚಿತ್ರ ಕಥೆಗಳು ಮತ್ತು ಕಾಮಿಕ್ಸ್ ಸಂಗ್ರಹ ಎಲ್ಲಾ ವಯಸ್ಸಿನ ಮಕ್ಕಳಿಗೆ ಒಂದು ಸಂವಾದಾತ್ಮಕ ತಾಣವಾಗಿದೆ"),
    ("train_kannadafullmale_00002", "ಮಕ್ಕಳು ಕೇವಲ ಭಾರತೀಯ ಸಂಸ್ಕೃತಿ ಮತ್ತು ಸಂಪ್ರದಾಯದ ಬಗ್ಗೆ ಜ್ಞಾನ ಪಡೆಯಲು ಆದರೆ ಪ್ರಸ್ತುತ ಘಟನೆಗಳ ಬಗ್ಗೆ ಮಾಹಿತಿ ಸುರಕ್ಷಿತ ಮೂಲವಾಗಿರಬಹುದು ಗುರಿಯನ್ನು ಹೊಂದಿದೆ"),
    ("train_kannadafullmale_00003", "ವಿಕ್ರಮ್ ಮತ್ತು ಭೇತಾಳ ನಂತಹ ಜನಪ್ರಿಯ ಕಥೆಗಳು, ಕೃಷ್ಣ, ಅಕ್ಬರ್ ಮತ್ತು ಬೀರ್ಬಲ್, ದೇವಿ ಇತ್ಯಾದಿ ಮತ್ತು ರಸ್ಕಿನ್ ಬಾಂಡ್ ಮುಂತಾದ ಲೇಖಕರು ಕಥೆಗಳು ಓದಿ"),
    ("train_kannadafullmale_00004", "ತಮ್ಮ ಅನುಪಮ ಪಾಂಡಿತ್ಯಕ್ಕೂ, ಜ್ಞಾನಕ್ಕೂ ಕಾಣಿಕೇಯಾಗಿ ಗೌರವ ಪೂರ್ವಕವಾಗಿ ಸಮರ್ಪಿಸಿಕೊಳ್ಳುತ್ತೇನೆ"),
    ("train_kannadafullmale_00005", "ಏನು ಬೇಕೋ ಅಪ್ಪಣೆ ಕೊಡಿ ಎಂದನು"),
    ("train_kannadafullmale_00006", "ಕ್ಷಮಿಸಬೇಕು ಪ್ರಭೂ, ಇಲ್ಲಿರುವವುಗಳಲ್ಲಿ ಯಾವುದೊಂದನ್ನೂ ಮುಟ್ಟೇನು ನ್ಯಾಯವಾದವುಗಳನ್ನು, ಧರ್ಮಬದ್ಧವಾದವುಗಳನ್ನು ಮಾತ್ರ ತೆಗೇದುಕೊಳ್ಳುತ್ತೇನೆ ಎಂದನು ಬ್ರಾಹ್ಮಣನು ಪುನರಾಲೋಚಿಸದೆ"),
    ("train_kannadafullmale_00007", "ಇಂತಹ ಸಂತೋಷಕರವಾದ ಸಂದರ್ಭದಲ್ಲಿ ದಾನಕ್ಕಾಗಿ ಬಂದ ಬ್ರಾಹ್ಮಣನನ್ನು ಬರಿಗೈಯಲ್ಲಿ ಕಳುಹಿಸುವುದು ತನ್ನ ಕೀರ್ತಿಗೆ ಭಂಗ ತರುತ್ತದಲ್ಲವೇ ಎಂದು ರಾಜನು ಮನಸ್ಸಿನಲ್ಲೇ ಮಂಥನ ಮಾಡಿಕೊಳ್ಳುತ್ತಿದ್ದನು"),
    ("train_kannadafullmale_00008", "ಬ್ರಾಹ್ಮಣನಿಗೇ ಹೇಗಾದರೂ ಸಂತೋಷ ಉಂಟುಮಾಡಬೇಕೇಂಬ ಉದ್ದೇಶದಿಂದ ಹಾಗೇ ಆಗಲಿ ಬ್ರಾಹ್ಮಣೋತ್ತಮಾ ನಾಳೆ ಬಂದು ನಿಮ್ಮ ಇಷ್ಟಾನುಸಾರವಾಗಿ ನಾನು ಸ್ವತಃ ಕಷ್ಟಪಟ್ಟು ಆರ್ಜಿಸಿದುದನ್ನು ಕಾಣಿಕೆಯಾಗಿ ಪಡೆದೊಯ್ಯಬಹುದು ಎಂದನು"),
    ("train_kannadafullmale_00009", "ನಸುನಗೇಯೊಡನೆ ಬ್ರಾಹ್ಮಣನು ನಿಧಾನವಾಗಿ ಬಾಗಿ ರಾಜನಿಗೇ ನಮಸ್ಕರಿಸಿ ಅಲ್ಲಿಂದ ಹೊರಟು ಹೋದನು"),
    ("train_kannadafullmale_00010", "ರಾಜನು ಅಂತಃಪುರಕ್ಕೇ ಹೋಗಿ ಹರಿದ ಬಟ್ಟೆಯೊಡನೆ ಬಡವನಂತೆ ವೇಷ ಹಾಕಿಕೊಂಡು ಅರಮನೆಯನ್ನು ಬಿಟ್ಟು ಕೇಲಸಕ್ಕಾಗಿ ಹುಡುಕುತ್ತಾ ಎಲ್ಲಿ ಕೆಲಸ ದೊರಕುತ್ತದೋ ಎಂದು ಒಂಟಿಯಾಗಿ ಕಾಲ್ನಡೆಯಲ್ಲಿ ಹೊರಟನು"),
    ("train_kannadafullmale_00011", "ಕೊನೆಗೇ ಅವನು ಸಮುದ್ರತೀರವನ್ನು ತಲುಪಿದನು, ಆಗತಾನೇ ಕೇಲವರು ಬೆಸ್ತರು ಮೀನು ಹಿಡಿಯಲು ದೋಣಿಗಳ ಮೇಲೆ ಸಮುದ್ರದೊಳಕ್ಕೇ ಹೋಗಲು ಸಿದ್ಧವಾಗುತ್ತಿದ್ದರು"),
    ("train_kannadafullmale_00012", "ಅವನ ವೇಷವನ್ನು ನೋಡಿ ಕನಿಕರಗೊಂಡ ಆ ವೃದ್ಧನು ಈ ದಿನ ನನಗೇ ಮೈಯಲ್ಲಿ ಚಟುವಟಿಕೇ ಇಲ್ಲ"),
    ("train_kannadafullmale_00013", "ಸಮ್ಮತವೆಂದು ಹೇಳಿದ ರಾಜನು ಬಲೆಯನ್ನು ತೆಗೇದುಕೊಂಡು ದೋಣಿಯಲ್ಲಿ ಸಮುದ್ರದೊಳಕ್ಕೇ ಸ್ವಲ್ಪದೂರ ಹೋಗಿ ಬಲೆಬೀಸಿದನು"),
    ("train_kannadafullmale_00014", "ಒಂದಾನೊಂದು ಕಾಲದಲ್ಲಿ ಚೋಳರಾಜ್ಯವನ್ನು ಧರ್ಮಾತ್ಮನಾದ ಒಬ್ಬ ರಾಜನು ಪಾಲಿಸುತ್ತಿದ್ದನು"),
    ("train_kannadafullmale_00015", "ಧರ್ಮ ಪರಿಪಾಲನೆಯನ್ನು ನಡೆಸುತ್ತಿದ್ದುದರಿಂದ ಪ್ರಜೆಗಳಿಗೆ ದಾನಧರ್ಮ ಮಾಡಬೇಕೇಂದರೆ ಕವಿ ಪಂಡಿತರನ್ನು ಸತ್ಕರಿಸುವುದೆಂದರೆ ಆ ರಾಜನಿಗೇ ತುಂಬಾ ಇಷ್ಟ"),
    ("train_kannadafullmale_00016", "ರಾಜಧಾನಿಯಲ್ಲಿ ನೂತನ ಸಂವತ್ಸರಾರಂಭದ ಸಮಾರಂಭಗಳನ್ನು ವಿಜೃಂಭಣೆಯಿಂದ ನಡೆಸುವುದು ಅಲ್ಲಿನ ಸಂಪ್ರದಾಯ"),
    ("train_kannadafullmale_00017", "ಯಥಾಪ್ರಕಾರ ಆ ವರ್ಷವೂ ಸಹಾ ಆಟಪಾಟಗಳೊಡನೆ ಸಮಾರಂಭಗಳು ಅದ್ಭುತವಾಗಿ ನಡೆದುವು"),
    ("train_kannadafullmale_00018", "ಅಲ್ಲಿದ್ದವುಗಳೆಲ್ಲವೂ ತೆರಿಗೆಯ ರೂಪದಲ್ಲಿ ಪ್ರಜೆಗಳಿಂದ ವಸೂಲು ಮಾಡಿದ, ಅಥವಾ ಪ್ರಜೆಗಳ ಹಣದಿಂದ ಕೊಂಡವೇ ಆಗಿದ್ದವು"),
    ("train_kannadafullmale_00019", "ರಾತ್ರಿಯೆಲ್ಲಾ ಸಮುದ್ರದಲ್ಲಿದ್ದರೂ ಹೆಚ್ಚಿಗೇ ಹಿಡಿಯುವೆನೆಂಬ ನಂಬಿಕೆ ಇರಲಿಲ್ಲ"),
    ("train_kannadafullmale_00020", "ರಾತ್ರಿಯೆಲ್ಲಾ ಕಾವಲುಕಾಯುತ್ತಿದ್ದ ಭಟರು ಬೆಳಗಿನ ಜಾವದಲ್ಲಿ ಮೈಮರೆತು ನಿದ್ದೆ ಮಾಡುತ್ತಿದ್ದರು"),
    ("train_kannadafullmale_00021", "ಅವನನ್ನು ನೋಡಿದೊಡನೆಯೇ ರಾಜನು ಬ್ರಾಹ್ಮಣೋತ್ತಮಾ, ನಿನ್ನೆ ರಾತ್ರಿ ನಾನು ಶ್ರಮಪಟ್ಟು ಒಂದು ತಾಮ್ರದ ನಾಣ್ಯ, ಮತ್ತು ಒಂದು ಕವಡೆಯನ್ನು ಸಂಪಾದಿಸಿದ್ದೇನೆ"),
    ("train_kannadafullmale_00022", "ಬ್ರಾಹ್ಮಣನು ಅವುಗಳನ್ನು ಅತ್ಯಂತ ಆನಂದದಿಂದ ಪಡೆದುಕೊಂಡು ಕಣ್ಣಿಗೇ ಒತ್ತಿಕೊಂಡು ಪ್ರಭುಗಳು ಬೆವರುಸುರಿಸಿ ಸಂಪಾದಿಸಿದ ಧನ ಇದು, ಇದಕ್ಕೆ ಬೆಲೆಕಟ್ಟಲಾಗದು"),
    ("train_kannadafullmale_00023", "ಮನೆಯ ಬಳಿ ಬ್ರಾಹ್ಮಣನ ಪತ್ನಿಯು ಎರಡನೆಯ ದಿನ ರಾಜದರ್ಶನಕ್ಕೆ ಹೋದ ಗಂಡನು, ಎಂತಹ ಬೆಲೆ ಬಾಳುವ ಬಹುಮಾನದೊಡನೆ ಹಿಂದಿರುಗುತ್ತಾನೋ ಎಂದು, ಅತ್ಯಂತ ಆಸಕ್ತಿಯಿಂದ ನಿರೀಕ್ಷಿಸುತ್ತಿದ್ದಳು"),
    ("train_kannadafullmale_00024", "ಗಂಡನು ನಸುನಗುತ್ತಾ ಬಂದೊಡನೆಯೇ ಖಂಡಿತವಾಗಿಯೂ ಬೆಲೆಯುಳ್ಳ ಬಹುಮಾನವನ್ನೇ ತಂದಿರಬೇಕೇಂಬ ನಂಬಿಕೆಯಿಂದ ಏನು ತಂದಿರಿ ಎಂದು ಕೇಳಿದಳು"),
    ("train_kannadafullmale_00025", "ಬ್ರಾಹ್ಮಣನು ಉತ್ತರೀಯದಲ್ಲಿ ಕಟ್ಟಿ ಸೊಂಟಕ್ಕೆ ಸಿಕ್ಕಿಸಿಕೊಂಡಿದ್ದ ತಾಮ್ರದ ನಾಣ್ಯ ಕವಡೆಗಳನ್ನು ತೆಗೆದು ಹೆಂಡತಿಯ ಅಂಗೈಯಲ್ಲಿಡುತ್ತಾ, ಇದು ರಾಜನು ಸ್ವತಃ ಕಷ್ಟಪಟ್ಟು ಸಂಪಾದಿಸಿದುದು"),
    ("train_kannadafullmale_00026", "ಖಜಾನೆಯಲ್ಲಿರುವ ಮುತ್ತು ರತ್ನಗಳಿಗಿಂತ, ಸ್ವರ್ಣಾಭರಣಗಳಿಗಿಂತ ಬೆಲೆಯುಳ್ಳದ್ದು"),
    ("train_kannadafullmale_00027", "ಇವುಗಳನ್ನೇ ಅತ್ಯಂತ ಅಮೂಲ್ಯವಾದುವುಗಳೆನ್ನುತ್ತೀಯಲ್ಲಾ, ನಿಮಗೇನಾಯಿತು, ಹೋಗುವಾಗ ಸರಿಯಾಗೇ ಇದ್ದಿರಲ್ಲಾ ಎನ್ನುತ್ತಾ ಬ್ರಾಹ್ಮಣನ ಪತ್ನಿಯು ಕೋಪದಿಂದ ಅವುಗಳನ್ನು ಬಾಗಿಲಿಗೆ ಬಿಸಾಡಿದಳು"),
    ("train_kannadafullmale_00028", "ತಾಮ್ರದ ನಾಣ್ಯವು ಉತ್ತರ ದಿಕ್ಕಿಗೂ ಕವಡೆಯು ದಕ್ಷಿಣ ದಿಕ್ಕಿಗೂ ಉರುಳಿ ಹೋಗಿ ಬಿದ್ದುವು"),
    ("train_kannadafullmale_00029", "ಬೆಳಗಾದ ಮೇಲೆ ಆ ಬ್ರಾಹ್ಮಣ ದಂಪತಿಗಳು ಬಾಗಿಲಿಗೆ ಬಂದು ಆ ದೃಶ್ಯವನ್ನು ನೋಡಿ ದಿಗ್ಭ್ರಾಂತಿಗೊಳಗಾದರು"),
    ("train_kannadafullmale_00030", "ಸಕಲ ಸೌಭಾಗ್ಯಗಳನ್ನೂ ಪಡೆದ ಗೃಹಸ್ಥರೆಂಬ ಕೀರ್ತಿಯನ್ನೂ ಪಡೆದುಕೊಂಡರು"),
    ("train_kannadafullmale_00031", "ದುಷ್ಟಬುದ್ಧಿಯ ಯೋಗಿಯು ರಾಜನ ಪ್ರಾರ್ಥನೆಯನ್ನು ಅಂಗೀಕರಿಸಿಕೊಂಡನು"),
    ("train_kannadafullmale_00032", "ಅವನಿಗೆ ರಾಜನು ಉದ್ಯಾನವನದಲ್ಲಿ ಎಲ್ಲಾ ವಸತಿಗಳನ್ನೂ ಏರ್ಪಡಿಸಿ ಕೊಟ್ಟು ಗುರುವಿನಂತೆ ಸೇವಿಸಲಿಕ್ಕೂ ತೊಡಗಿದನು"),
    ("train_kannadafullmale_00033", "ಸುಮನಸ ಕುಮಾರನಿಗೆ ಏಳನೇ ವಯಸ್ಸು ಬರುತ್ತಿದ್ದಾಗ ರೇಣುಕ ರಾಜನಿಗೆ ಸಾಮಂತ ರಾಜರೊಂದಿಗೆ ಯುದ್ಧಕ್ಕೆ ಹೋಗಬೇಕಾದ ಪ್ರಸಂಗ ಬಂತು"),
    ("train_kannadafullmale_00034", "ರಾಜನು ವಿಜಯಿಯಾಗಿ ಬರುತ್ತಿರುವುದಾಗಿ ತಿಳಿದ ಕಪಟಯೋಗಿಯು ತನ್ನ ಕಮಂಡಲ ವನ್ನೂ ಪೀಠವನ್ನೂ ತಾನೇ ಒಡೆದು ಹಾಕಿ ಆಶ್ರಮದ ಸುತ್ತಲೂ ಕಸ ಕಡ್ಡಿಗಳನ್ನು ಚೆಲ್ಲಿ ಒಂದು ಮೂಲೆಯಲ್ಲಿ ಮುಲುಗುತ್ತಾ ಮಲಗಿ ಕೊಂಡನು"),
    ("train_kannadafullmale_00035", "ಒಮ್ಮೆ ಹಿಮಾಲಯ ಪರ್ವತದಲ್ಲಿದ್ದ ಮಹಾರಕ್ಷಿತನೆಂಬ ತಪಸ್ವಿಯು ತನ್ನ ಐವರು ಶಿಷ್ಯರೊಂದಿಗೆ ದೇಶ ಸಂಚಾರಕ್ಕೆ ಹೊರಟನು"),
    ("train_kannadafullmale_00036", "ಮಾರ್ಗದಲ್ಲಿ ವಿಶ್ರಾಂತಿ ಪಡೆಯುವಾಗ ತಮಗಷ್ಟು ಸತ್ಕಾರ ವನ್ನು ಮಾಡಿದ ರಾಜನಿಗೆ ಇನ್ನೂ ಸಂತಾನವಿಲ್ಲವೆಂಬ ಸಂಗತಿಯನ್ನು ಶಿಷ್ಯರು ಹೇಳಿದಾಗ ಮಹಾರಕ್ಷಿತನು ರೇಣುಕ ರಾಜನಿಗೆ ದೈವಾಂಶದ ಕುಮಾರನೊಬ್ಬನು ಹುಟ್ಟುತ್ತಾನೆ"),
    ("train_kannadafullmale_00037", " ತಮ್ಮ ಗುರುವಾದ ಮಹಾರಕ್ಷಿತನು ವಾಕ್ಶುದ್ಧಿ ಯುಳ್ಳವನೆಂದು ಶಿಷ್ಯರಿಗೆಲ್ಲಾ ಗೊತ್ತಿತ್ತು " ),
    ("train_kannadafullmale_00038", " ಈ ಶುಭವಾರ್ತೆ ಕೇಳಿ ಆನಂದ ಭರಿತನಾದ ರಾಜನು ಆ ಯೋಗಿಯನ್ನು ನಿಲ್ಲಿಸಿ, ಮಾಹಾತ್ಮಾ, ತಾವು ಸಾಮಾನ್ಯದವರಲ್ಲ, ದಿವ್ಯಚಕ್ಷುಗಳು, ಇಲ್ಲೇ ಇದ್ದು ನಮ್ಮ ಸೇವೆ ಸ್ವೀಕರಿಸುವಂತೆ ಕೇಳಿ ಕೊಳ್ಳುತ್ತೇನೆ ಎಂದು ಪ್ರಾರ್ಥಿಸಿದನು " ),
    ("train_kannadafullmale_00039", "ರಾಜನು ತನ್ನ ತಪ್ಪಿಗೆ ಬಹಳ ದುಃಖಿಸುತ್ತ ಮಗನೊಂದಿಗೆ ಕಂದಾ, ನನ್ನ ದುಡುಕುತನವನ್ನು ಮನ್ನಿಸು " ),
    ("train_kannadafullmale_00040", "ಈ ಮೊದಲು ಕಪಟ ಯೋಗಿಯ ಮಾತು ಕೇಳಿ ದುಡುಕಿ ನನ್ನ ಶಿರಚ್ಛೇದನಕ್ಕೆ ಆಜ್ಞೆ ಮಾಡಿ ದಂತೆಯೇ ಹುಡುಗನಾದ ನನ್ನ ಮೇಲೆ ರಾಜ್ಯ ಭಾರ ಹಾಕಲು ನೋಡುವುದೂ ಸಹ ದುಡುಕು ಬುದ್ಧಿಯೇ,ನಾನು ಈಗಲೇ ನಿಮ್ಮ ರಾಜ್ಯವನ್ನು ಬಿಟ್ಟೇ ಹೊರಟು ಹೋಗುತ್ತೇನೆ ಎಂದನು ")]

metadata = pd.DataFrame(metadata[1:], columns=metadata[0])

# Display the DataFrame
print(metadata)

                             ID  \
0   train_kannadafullmale_00001   
1   train_kannadafullmale_00002   
2   train_kannadafullmale_00003   
3   train_kannadafullmale_00004   
4   train_kannadafullmale_00005   
5   train_kannadafullmale_00006   
6   train_kannadafullmale_00007   
7   train_kannadafullmale_00008   
8   train_kannadafullmale_00009   
9   train_kannadafullmale_00010   
10  train_kannadafullmale_00011   
11  train_kannadafullmale_00012   
12  train_kannadafullmale_00013   
13  train_kannadafullmale_00014   
14  train_kannadafullmale_00015   
15  train_kannadafullmale_00016   
16  train_kannadafullmale_00017   
17  train_kannadafullmale_00018   
18  train_kannadafullmale_00019   
19  train_kannadafullmale_00020   
20  train_kannadafullmale_00021   
21  train_kannadafullmale_00022   
22  train_kannadafullmale_00023   
23  train_kannadafullmale_00024   
24  train_kannadafullmale_00025   
25  train_kannadafullmale_00026   
26  train_kannadafullmale_00027   
27  train_kannadaful

In [3]:
output_folder = "generated_wavs_kannada"
os.makedirs(output_folder, exist_ok=True)

In [4]:
import scipy
# Iterate through each row of the metadata to generate and save audio
for index, row in metadata.iterrows():
    text = row['Text']
    wav_name = f"{row['ID']}.wav"  # Add the .wav extension
    output_path = os.path.join(output_folder, wav_name)

    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt")

    # Generate the audio waveform
    with torch.no_grad():
        output = model(**inputs).waveform

    # Save the generated audio as a .wav file
    scipy.io.wavfile.write(output_path, rate=model.config.sampling_rate, data=output.squeeze().numpy())

    print(f"Generated and saved: {output_path}")

Generated and saved: generated_wavs_kannada\train_kannadafullmale_00001.wav
Generated and saved: generated_wavs_kannada\train_kannadafullmale_00002.wav
Generated and saved: generated_wavs_kannada\train_kannadafullmale_00003.wav
Generated and saved: generated_wavs_kannada\train_kannadafullmale_00004.wav
Generated and saved: generated_wavs_kannada\train_kannadafullmale_00005.wav
Generated and saved: generated_wavs_kannada\train_kannadafullmale_00006.wav
Generated and saved: generated_wavs_kannada\train_kannadafullmale_00007.wav
Generated and saved: generated_wavs_kannada\train_kannadafullmale_00008.wav
Generated and saved: generated_wavs_kannada\train_kannadafullmale_00009.wav
Generated and saved: generated_wavs_kannada\train_kannadafullmale_00010.wav
Generated and saved: generated_wavs_kannada\train_kannadafullmale_00011.wav
Generated and saved: generated_wavs_kannada\train_kannadafullmale_00012.wav
Generated and saved: generated_wavs_kannada\train_kannadafullmale_00013.wav
Generated an

In [5]:
import os
import csv
import librosa
import numpy as np
from jiwer import cer, wer
import parselmouth

def compute_metrics(original_folder, generated_folder, output_csv):
    metrics = {
        "File": [], "MCD": [], "LSD": [], "SNR": [],
        "Pitch RMSE": [], "Duration Difference": [], "CER": [], "WER": []
    }
    
    original_files = sorted(os.listdir(original_folder))
    generated_files = sorted(os.listdir(generated_folder))
    
    for orig_file, gen_file in zip(original_files, generated_files):
        orig_path = os.path.join(original_folder, orig_file)
        gen_path = os.path.join(generated_folder, gen_file)
        
        # Load audio files
        orig_audio, orig_sr = librosa.load(orig_path, sr=None)
        gen_audio, gen_sr = librosa.load(gen_path, sr=None)
        duration_diff = abs(len(orig_audio) / orig_sr - len(gen_audio) / gen_sr)
        
        # Resample if needed
        if orig_sr != gen_sr:
            gen_audio = librosa.resample(gen_audio, gen_sr, orig_sr)
            gen_sr = orig_sr
        
        # Align audio lengths
        min_length = min(len(orig_audio), len(gen_audio))
        orig_audio = orig_audio[:min_length]
        gen_audio = gen_audio[:min_length]
        
        # Compute spectrograms
        orig_mel = librosa.feature.melspectrogram(y=orig_audio, sr=orig_sr)
        gen_mel = librosa.feature.melspectrogram(y=gen_audio, sr=gen_sr)
        
        # Align spectrogram shapes
        min_frames = min(orig_mel.shape[1], gen_mel.shape[1])
        orig_mel = orig_mel[:, :min_frames]
        gen_mel = gen_mel[:, :min_frames]
        
        # Compute metrics
        mcd = np.mean(np.abs(orig_mel - gen_mel))  # Simplified
        lsd = np.mean(np.abs(librosa.amplitude_to_db(orig_mel) - librosa.amplitude_to_db(gen_mel)))
        noise = orig_audio - gen_audio
        snr = 10 * np.log10(np.sum(orig_audio ** 2) / np.sum(noise ** 2))
        orig_pitch = parselmouth.Sound(orig_path).to_pitch().selected_array["frequency"]
        gen_pitch = parselmouth.Sound(gen_path).to_pitch().selected_array["frequency"]
        min_pitch_length = min(len(orig_pitch), len(gen_pitch))
        pitch_rmse = np.sqrt(np.mean((orig_pitch[:min_pitch_length] - gen_pitch[:min_pitch_length]) ** 2))
        # duration_diff = abs(len(orig_audio) / orig_sr - len(gen_audio) / gen_sr)
        orig_text = os.path.splitext(orig_file)[0]  # Assumes filename contains transcription
        gen_text = os.path.splitext(gen_file)[0]  # Assumes filename contains transcription
        char_error_rate = cer(orig_text, gen_text)
        word_error_rate = wer(orig_text, gen_text)
        
        # Append to metrics
        metrics["File"].append(orig_file)
        metrics["MCD"].append(mcd)
        metrics["LSD"].append(lsd)
        metrics["SNR"].append(snr)
        metrics["Pitch RMSE"].append(pitch_rmse)
        metrics["Duration Difference"].append(duration_diff)
        metrics["CER"].append(char_error_rate)
        metrics["WER"].append(word_error_rate)
    
    # Write metrics to a CSV file
    with open(output_csv, mode='w', newline='') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(metrics.keys())  # Write header
        writer.writerows(zip(*metrics.values()))  # Write rows
    
    print(f"Metrics saved to {output_csv}")



# Folders containing original and generated wav files
original_folder = "D:/Wav2Lip-master/TTS Evaluation/wav_kannada"
generated_folder = "D:/Wav2Lip-master/TTS Evaluation/generated_wavs_kannada"

# Output CSV file
output_csv = "tts_model_evaluation_kannada.csv"

# Compute and save metrics
compute_metrics(original_folder, generated_folder, output_csv)


C:\Users\satvi\AppData\Local\Temp\ipykernel_5116\837590111.py:28: FutureWarning: Pass orig_sr=16000, target_sr=48000 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  gen_audio = librosa.resample(gen_audio, gen_sr, orig_sr)
C:\Users\satvi\AppData\Local\Temp\ipykernel_5116\837590111.py:28: FutureWarning: Pass orig_sr=16000, target_sr=48000 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  gen_audio = librosa.resample(gen_audio, gen_sr, orig_sr)


Metrics saved to tts_model_evaluation_kannada.csv
